# Banc d'essai — réseaux pour l'ESP32

Un essai = une configuration (cellule 1). Il est entraîné ici en interactif (numpy, processeur),
enregistré dans le journal (`banc/resultats/`), puis comparé aux autres essais, y compris ceux
lancés en lot sur la carte graphique (cellule 6). Un essai du lot se rejoue à l'identique ici.

**Règle** : on choisit sur la *validation* ; le *test* ne sert qu'une fois, pour le réseau retenu (cellule 7).

Coût ESP estimé à partir des mesures de `esp32/ESSAIS.md` (5,3 cycles par multiplication-addition,
436 cycles par sigmoïde…). Moteur C++ actuel : couches denses, sigmoïde / ReLU / tanh, softmax.

In [ ]:
# 1. PARAMÈTRES — le seul endroit à modifier

# --- Tâche : "mnist" (options : facteur 1, 2, 4) ou "signal" (signal 1D synthétique)
TACHE         = "mnist"
TACHE_OPTIONS = {"facteur": 2}
N_IMAGES      = None              # sous-ensemble d'entraînement (None : tout)

# --- Réseau : couches cachées ; la sortie softmax est ajoutée d'office
#   {"type": "dense", "n": 30, "act": "sigmoide" | "relu" | "tanh"}
#   {"type": "conv1d", "canaux": 8, "noyau": 7, "pas": 1, "act": "relu"}   (entrées 1D)
#   {"type": "maxpool1d", "taille": 2}
COUCHES = [{"type": "dense", "n": 30, "act": "sigmoide"}]
INIT    = "auto"                  # auto : 1/sqrt(n), ou sqrt(2/n) pour ReLU

# --- Apprentissage
EPOQUES      = 30
BATCH        = 10                 # m
ETA          = 0.5                # η maximal
PLANNING     = "cosinus"          # "cosinus" ou "constant"
ECHAUFFEMENT = 1.0                # époques de montée linéaire (cosinus)
ETA_FIN      = 0.0
LMBDA        = 5.0                # L2, poids seulement
GRAINE       = 1                  # poids de départ
GRAINE_ORDRE = 0                  # ordre des images
PRECISION    = "float32"
NOTE         = ""

# --- Affichage
AFFICHER_TOUS_LES = 1             # époques entre deux mises à jour des courbes

In [ ]:
# 2. PRÉPARATION — une fois par session
import os, sys, time
RACINE = os.path.abspath("..") if os.path.basename(os.getcwd()) == "banc" else os.getcwd()
sys.path.insert(0, RACINE)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from banc import essai, analyses, lot, config
from banc.couts import couts
from banc.export_esp import vers_esp

def configuration():
    return config.completer(tache=TACHE, tache_options=TACHE_OPTIONS, n_images=N_IMAGES, couches=COUCHES,
                            init=INIT, epoques=EPOQUES, batch=BATCH, eta=ETA, planning=PLANNING,
                            echauffement=ECHAUFFEMENT, eta_fin=ETA_FIN, lmbda=LMBDA, graine=GRAINE,
                            graine_ordre=GRAINE_ORDRE, precision=PRECISION, note=NOTE)

cfg = configuration()
c = couts(cfg)
print(config.resume(cfg), "  id", config.identifiant(cfg))
print("Poids : %d (%.1f Ko) · mult.-add. : %d · ESP estimé : %.0f µs · déployable : %s"
      % (c["params"], c["flash_ko"], c["macs"], c["us_esp"], "oui" if c["deployable"] else "pas encore"))
print("Essais au journal :", len(essai.journal()))

In [ ]:
# 3. LANCER UN ESSAI (interactif)
cfg = configuration()
temps = {"calcul": 0.0, "affichage": 0.0}

def afficher(info):
    if info["epoque"] % AFFICHER_TOUS_LES and info["epoque"] != cfg["epoques"]:
        return
    t0 = time.perf_counter()
    h = info["historique"]
    clear_output(wait=True)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
    a1.plot([e["epoque"] for e in h], [e["validation"] for e in h], "-o", ms=3, color="#2a78d6")
    a1.set_xlabel("époque"); a1.set_ylabel("validation (%)"); a1.grid(True, color="#e6e5e0")
    a1.set_title("%s — %.2f %%" % (config.resume(cfg), h[-1]["validation"]), fontsize=9)
    a2.plot([e["epoque"] for e in h], [e["eta"] for e in h], "-", color="#eb6834")
    a2.set_xlabel("époque"); a2.set_ylabel("η en fin d'époque"); a2.grid(True, color="#e6e5e0")
    plt.show()
    temps["affichage"] += time.perf_counter() - t0
    print("époque %d/%d · calcul %.1f s · affichage %.1f s"
          % (info["epoque"], cfg["epoques"], info["duree_calcul"], temps["affichage"]))

res = essai.lancer(cfg, rappel=afficher)
print("Validation : %.2f %% (meilleure %.2f %%) · id %s · ESP estimé %.0f µs"
      % (res["validation"], res["meilleure_validation"], res["id"], res["couts"]["us_esp"]))

In [ ]:
# 4. ANALYSER CET ESSAI (validation)
print(analyses.confiance(res))
plt.show()
analyses.confusion(res); plt.show()
analyses.erreurs(res, n=16); plt.show()

In [ ]:
# 5. COMPARER LES ESSAIS DU JOURNAL
tous = essai.journal({"tache": TACHE, "tache_options": TACHE_OPTIONS})
print(len(tous), "essais pour cette tâche")
fig, ax = plt.subplots(figsize=(9, 5))
front = analyses.frontiere(tous, cout="us_esp", ax=ax)       # ou "params", "flash_ko", "macs"
plt.show()
for r in front:
    print("%6.2f %%  %6.0f µs  %s" % (r["validation"], r["couts"]["us_esp"], r["resume"]))
analyses.courbes(sorted(tous, key=lambda r: -r["validation"])[:6]); plt.show()
analyses.par_graines(tous);

In [ ]:
# 6. LOT — beaucoup d'essais sur la carte graphique (ou le processeur à défaut)
LANCER_LOT = False                # passer à True pour lancer

base = configuration()
cfgs = lot.grille(base,
                  eta=[0.1, 0.25, 0.5, 1.0],
                  lmbda=[0.0, 2.0, 5.0],
                  couches=[[{"type": "dense", "n": n, "act": a}] for n in (10, 20, 30, 50) for a in ("sigmoide", "relu")])
cfgs = lot.avec_graines(cfgs, graines=(1, 2, 3))            # trois graines : mesurer le bruit
print(len(cfgs), "configurations")
if LANCER_LOT:
    resultats = lot.lancer_lot(cfgs, taille_paquet=256)

In [ ]:
# 7. RETENIR UN RÉSEAU ET L'EXPORTER VERS L'ESP
#    Le test est mesuré ici, une seule fois, puis inscrit dans le journal et dans le modèle.
ID_RETENU = None                  # identifiant du journal, ex. "3f2a9c1e5b"

if ID_RETENU:
    chemin = vers_esp(ID_RETENU)
    # puis, dans un terminal : python esp32\\python\\export.py <chemin>  (headers C++ + autocontrôles)